In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os
import sys 
project_path = os.path.join(os.getcwd(), '..', '..')
sys.path.append(project_path)

from utils.transformations import reusable

###Dim User



In [0]:
df_user  = spark.readStream.format("cloudFiles").option("cloudFiles.format", "parquet").option("cloudFiles.schemaLocation","abfss://silver@storageazureproject11.dfs.core.windows.net/DimUser/Checkpoint").option("schemaEvolutionMode", "addNewColumns").load("abfss://bronze@storageazureproject11.dfs.core.windows.net/DimUser")

In [0]:
df = df_user.withColumn("user_name", upper(col("user_name")))

In [0]:
df_user_obj = reusable()
df1 = df_user_obj.dropColumns(df,['_rescued_data'] )
df2 = df1.dropDuplicates(['user_id'])

In [0]:
df2.writeStream.format('Delta')\
    .outputMode("append")\
    .option("checkPointLocation","abfss://silver@storageazureproject11.dfs.core.windows.net/DimUser/Checkpoint")\
    .trigger(once=True)\
    .option("path","abfss://silver@storageazureproject11.dfs.core.windows.net/DimUser/Data")\
    .toTable("project_catalog.silver.DimUser")

###DimArtist

In [0]:
df_art = spark.readStream.format('cloudFiles')\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation","abfss://silver@storageazureproject11.dfs.core.windows.net/DimArtist/Checkpoint")\
    .option("schemaEvolutionMode","addNewColumns")\
    .load("abfss://bronze@storageazureproject11.dfs.core.windows.net/DimArtist")

In [0]:
df_art_obj = reusable()
df1 = df_art_obj.dropColumns(df_art,['_rescued_data'] )
df2 = df1.dropDuplicates(['artist_id'])

In [0]:
df2.writeStream.format('Delta')\
    .outputMode("append")\
    .option("checkPointLocation","abfss://silver@storageazureproject11.dfs.core.windows.net/DimArtist/Checkpoint")\
    .trigger(once=True)\
    .option("path","abfss://silver@storageazureproject11.dfs.core.windows.net/DimArtist/Data")\
    .toTable("project_catalog.silver.DimArtist")

###DimTrack

In [0]:
df_trk = spark.readStream.format('cloudFiles')\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation","abfss://silver@storageazureproject11.dfs.core.windows.net/DimTrack/Checkpoint")\
    .option("schemaEvolutionMode","addNewColumns")\
    .load("abfss://bronze@storageazureproject11.dfs.core.windows.net/DimTrack")

In [0]:
df_trk1 = df_trk.withColumn("durationFlag", when(col('duration_sec') > 150 , "low")\
                                            .when(col('duration_sec') > 300 , "medium")\
                                            .otherwise("high"))

df_trk2 = df_trk1.withColumn("track_name", regexp_replace(col('track_name'), "-", " "))


In [0]:
df_trk_obj = reusable()
df1 = df_trk_obj.dropColumns(df_trk2,['_rescued_data'] )

In [0]:
df1.writeStream.format('Delta')\
    .outputMode("append")\
    .option("checkPointLocation","abfss://silver@storageazureproject11.dfs.core.windows.net/DimTrack/Checkpoint")\
    .trigger(once=True)\
    .option("path","abfss://silver@storageazureproject11.dfs.core.windows.net/DimTrack/Data")\
    .toTable("project_catalog.silver.DimTrack")

###DimDate

In [0]:
df = spark.readStream.format('cloudFiles')\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation","abfss://silver@storageazureproject11.dfs.core.windows.net/DimDate/Checkpoint")\
    .option("schemaEvolutionMode","addNewColumns")\
    .load("abfss://bronze@storageazureproject11.dfs.core.windows.net/DimDate")

In [0]:
df_obj = reusable()
df1 = df_obj.dropColumns(df,['_rescued_data'] )

In [0]:
df1.writeStream.format('Delta')\
    .outputMode("append")\
    .option("checkPointLocation","abfss://silver@storageazureproject11.dfs.core.windows.net/DimDate/Checkpoint")\
    .trigger(once=True)\
    .option("path","abfss://silver@storageazureproject11.dfs.core.windows.net/DimDate/Data")\
    .toTable("project_catalog.silver.DimDate")

###FactStream

In [0]:
df_fact = spark.readStream.format('cloudFiles')\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation","abfss://silver@storageazureproject11.dfs.core.windows.net/FactStream/Checkpoint")\
    .option("schemaEvolutionMode","addNewColumns")\
    .load("abfss://bronze@storageazureproject11.dfs.core.windows.net/FactStream")

In [0]:
df_obj = reusable()
df1 = df_obj.dropColumns(df_fact,['_rescued_data'] )

In [0]:
df1.writeStream.format('Delta')\
    .outputMode("append")\
    .option("checkPointLocation","abfss://silver@storageazureproject11.dfs.core.windows.net/FactStream/Checkpoint")\
    .trigger(once=True)\
    .option("path","abfss://silver@storageazureproject11.dfs.core.windows.net/FactStream/Data")\
    .toTable("project_catalog.silver.FactStream")